In [0]:
# Passo 3.1 - Silver: separando código e nome da função
from pyspark.sql import functions as F

df_bronze_despesas = spark.table("workspace.gastos_municipais.bronze_despesas")

df_silver_despesas = (
    df_bronze_despesas
    .withColumn("codigo_funcao", F.split(F.col("conta"), " - ").getItem(0))
    .withColumn("nome_funcao", F.split(F.col("conta"), " - ").getItem(1))
    .withColumn("ano", F.lit(2025))
    .select("cod_ibge", "ano", "codigo_funcao", "nome_funcao", "valor")
)

df_silver_despesas.show(5, truncate=False)

In [0]:
# Passo 3.2 - Persistindo a tabela Silver de despesas por função
df_silver_despesas.write.mode("overwrite").saveAsTable("workspace.gastos_municipais.gold_despesas_indicadores")

spark.sql("SHOW TABLES IN workspace.gastos_municipais").show(truncate=False)

In [0]:
# Passo 3.3 - Silver: limpando a receita (removendo, com documentação, os municípios sem valor)
df_bronze_receitas = spark.table("workspace.gastos_municipais.bronze_receitas")

df_gold_despesas_indicadores = (
    df_bronze_receitas
    .withColumn("ano", F.lit(2025))
    .filter(F.col("receita_total").isNotNull())
    .select("cod_ibge", "ano", "receita_total")
)

print(f"Linhas originais: {df_bronze_receitas.count()} | Linhas após remover nulos: {df_gold_despesas_indicadores.count()}")

df_gold_despesas_indicadores.write.mode("overwrite").saveAsTable("workspace.gastos_municipais.gold_despesas_indicadores")

In [0]:
# Passo 3.4 - Criando a tabela dimensão de municípios
df_gold_despesas_indicadores = spark.table("workspace.gastos_municipais.bronze_municipios_amostra")

df_gold_despesas_indicadores.write.mode("overwrite").saveAsTable("workspace.gastos_municipais.gold_despesas_indicadores")

spark.sql("SHOW TABLES IN workspace.gastos_municipais").show(truncate=False)

In [0]:
# Passo 4.1 - Gold: juntando despesas + receita + dimensão de município
df_despesas = spark.table("workspace.gastos_municipais.gold_despesas_indicadores")
df_receitas = spark.table("workspace.gastos_municipais.gold_despesas_indicadores")
df_dim = spark.table("workspace.gastos_municipais.gold_despesas_indicadores")

df_gold = (
    df_despesas
    .join(df_receitas, on=["cod_ibge", "ano"], how="inner")
    .join(df_dim, on="cod_ibge", how="inner")
    .withColumn("pct_receita", F.round(F.col("valor") / F.col("receita_total") * 100, 2))
    .withColumn("valor_per_capita", F.round(F.col("valor") / F.col("populacao_2025"), 2))
)

print(f"Total de linhas: {df_gold.count()}")
df_gold.show(5, truncate=False)

In [0]:
# Passo 4.2 - Conferindo quantos municípios distintos restaram após o join
print(f"Municípios distintos em df_gold: {df_gold.select('cod_ibge').distinct().count()}")

In [0]:
# Passo 4.3 - Persistindo a tabela Gold
df_gold.write.mode("overwrite").saveAsTable("workspace.gastos_municipais.gold_despesas_indicadores")

print("Tabela gold_despesas_indicadores salva com sucesso.")

In [0]:
# Passo 5.1 - Pergunta 2: gasto per capita em Saúde e Educação por região
df_q2 = (
    df_gold
    .filter(F.col("codigo_funcao").isin("10", "12"))  # 10 = Saúde, 12 = Educação
    .groupBy("regiao", "nome_funcao")
    .agg(F.round(F.avg("valor_per_capita"), 2).alias("gasto_per_capita_medio"))
    .orderBy("nome_funcao", F.desc("gasto_per_capita_medio"))
)

df_q2.show(20, truncate=False)

In [0]:
# Passo 5.2 - Comparando com a média ponderada pela população (soma do gasto / soma da população)
df_q2_ponderada = (
    df_gold
    .filter(F.col("codigo_funcao").isin("10", "12"))
    .groupBy("regiao", "nome_funcao")
    .agg(F.round(F.sum("valor") / F.sum("populacao_2025"), 2).alias("gasto_per_capita_ponderado"))
    .orderBy("nome_funcao", F.desc("gasto_per_capita_ponderado"))
)

df_q2_ponderada.show(20, truncate=False)

In [0]:
# Passo 5.3 - Ranking das capitais por gasto per capita em Saúde e Educação
df_ranking_capitais = (
    df_gold
    .filter((F.col("eh_capital") == True) & (F.col("codigo_funcao").isin("10", "12")))
    .select("municipio", "uf", "regiao", "nome_funcao", "valor_per_capita")
    .orderBy("nome_funcao", F.desc("valor_per_capita"))
)

df_ranking_capitais.show(60, truncate=False)

In [0]:
# Passo 5.4 - Ranking geral de todos os municípios da amostra em Educação
df_educacao = df_gold.filter(F.col("codigo_funcao") == "12").select(
    "municipio", "uf", "regiao", "porte", "valor_per_capita"
)

print("TOP 10 - Maior gasto per capita em Educação:")
df_educacao.orderBy(F.desc("valor_per_capita")).show(10, truncate=False)

print("BOTTOM 10 - Menor gasto per capita em Educação:")
df_educacao.orderBy("valor_per_capita").show(10, truncate=False)

In [0]:
# Passo 5.5 - Pergunta 1: % médio da receita gasto em cada área (todos os municípios da amostra)
df_q1 = (
    df_gold
    .groupBy("nome_funcao")
    .agg(F.round(F.avg("pct_receita"), 2).alias("pct_receita_medio"))
    .orderBy(F.desc("pct_receita_medio"))
)

df_q1.show(30, truncate=False)

In [0]:
# Passo 5.6 - Top 5 áreas de gasto (% da receita) por porte de município
from pyspark.sql.window import Window

df_q1_porte = (
    df_gold
    .groupBy("porte", "nome_funcao")
    .agg(F.round(F.avg("pct_receita"), 2).alias("pct_receita_medio"))
)

# Window: particiona por porte, ordena por % dentro de cada partição, numera as linhas
janela = Window.partitionBy("porte").orderBy(F.desc("pct_receita_medio"))

df_top5_por_porte = (
    df_q1_porte
    .withColumn("posicao", F.row_number().over(janela))
    .filter(F.col("posicao") <= 5)
    .orderBy("porte", "posicao")
)

df_top5_por_porte.show(30, truncate=False)

In [0]:
# Passo 6.1 - Documentando a tabela gold_despesas_indicadores no Unity Catalog
spark.sql("COMMENT ON TABLE workspace.gastos_municipais.gold_despesas_indicadores IS 'Tabela dimensão com atributos de cada município da amostra: identificação, localização, porte populacional e se é capital.'")

spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN cod_ibge COMMENT 'Código IBGE de 7 dígitos, identificador único do município (chave de junção com as demais tabelas)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN municipio COMMENT 'Nome do município'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN uf COMMENT 'Sigla da Unidade Federativa (2 letras)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN populacao_2025 COMMENT 'População estimada em 1º de julho de 2025, segundo o IBGE'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN porte COMMENT 'Classificação por porte populacional: Pequeno (até 20 mil), Médio (20-100 mil), Grande (100-500 mil), Metrópole (acima de 500 mil) ou Capital'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN regiao COMMENT 'Região geográfica do Brasil (Norte, Nordeste, Centro-Oeste, Sudeste, Sul)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN eh_capital COMMENT 'Indica se o município é capital de estado ou do Distrito Federal'")

print("Comentários adicionados!")

In [0]:
# Passo 6.2 - Documentando a tabela gold_despesas_indicadores no Unity Catalog

spark.sql("COMMENT ON TABLE workspace.gastos_municipais.gold_despesas_indicadores IS 'Tabela fato com o valor da despesa liquidada de cada município, por função de governo (área de gasto), referente ao exercício de 2025.'")

spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN cod_ibge COMMENT 'Código IBGE de 7 dígitos, identificador único do município (chave de junção com as demais tabelas)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN ano COMMENT 'Exercício financeiro de referência da despesa'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN codigo_funcao COMMENT 'Nome da função de governo, conforme classificação orçamentária do Tesouro Nacional (SICONFI, Anexo I-E da DCA)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN nome_funcao COMMENT 'Nome da função de governo (área de gasto) correspondente ao código, ex: Saúde, Educação, Administração'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN valor COMMENT 'Valor da despesa liquidada (R$) no exercício, para aquela função de governo naquele município'")

print("Comentários adicionados!")

In [0]:
# Passo 6.3 - Documentando a tabela gold_despesas_indicadores no Unity Catalog

spark.sql("COMMENT ON TABLE workspace.gastos_municipais.gold_despesas_indicadores IS 'Tabela fato com o valor da despesa liquidada de cada município, por função de governo (área de gasto), referente ao exercício de 2025.'")

spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN cod_ibge COMMENT 'Código IBGE de 7 dígitos, identificador único do município (chave de junção com as demais tabelas)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN ano COMMENT 'Exercício financeiro de referência da receita'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN receita_total COMMENT 'Valor total da receita bruta realizada (R$) no exercício, antes de deduções como FUNDEB (SICONFI, Anexo I-C da DCA, coluna Receitas Brutas Realizadas)'")


print("Comentários adicionados!")

In [0]:
# Passo 6.4 - Documentando a tabela gold_despesas_indicadores no Unity Catalog

spark.sql("COMMENT ON TABLE workspace.gastos_municipais.gold_despesas_indicadores IS 'Tabela Gold (analítica) com a junção de despesas por função, receita total e atributos do município, já com os indicadores de % da receita e gasto per capita calculados. Base usada para responder as perguntas do objetivo do trabalho.'")

spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN cod_ibge COMMENT 'Código IBGE de 7 dígitos, identificador único do município (chave de junção com as demais tabelas)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN ano COMMENT 'Exercício financeiro de referência (2025)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN codigo_funcao COMMENT 'Código de 2 dígitos da função de governo (área de gasto)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN nome_funcao COMMENT 'Nome da função de governo (área de gasto), ex: Saúde, Educação'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN valor COMMENT 'Valor da despesa liquidada (R$) para aquela função de governo'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN receita_total COMMENT 'Valor total da receita bruta realizada (R$) do município no exercício'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN municipio COMMENT 'Nome do município'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN uf COMMENT 'Sigla da Unidade Federativa (2 letras)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN populacao_2025 COMMENT 'População estimada em 1º de julho de 2025 (IBGE)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN porte COMMENT 'Classificação por porte populacional: Pequeno (até 20 mil), Médio (20-100 mil), Grande (100-500 mil), Metrópole (acima de 500 mil) ou Capital'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN regiao COMMENT 'Região geográfica do Brasil (Norte, Nordeste, Centro-Oeste, Sudeste, Sul)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN eh_capital COMMENT 'Indica se o município é capital de estado ou do Distrito Federal'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN pct_receita COMMENT 'Percentual da receita total do município gasto naquela função de governo (valor / receita_total × 100)'")
spark.sql("ALTER TABLE workspace.gastos_municipais.gold_despesas_indicadores ALTER COLUMN valor_per_capita COMMENT 'Valor da despesa naquela função de governo dividido pela população estimada do município (R$ por habitante)'")

print("Comentários adicionados!")